# Preparació de les dades

Modifica les dades perquè els algorismes de ML puguin aprendre correctament a partir d'elles.

### Imports

In [ ]:
# Importa les biblioteques, funcions, objectes... necessaris
# =====================================================================
# IMPORTS Y CARGA DEL DATASET
# =====================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


### Carrega el dataset

In [ ]:
df = pd.read_csv('../data/house_pricing.csv')

# Eliminamos la columna duplicada por error tipográfico antes de la separación
if 'Electtrical' in df.columns:
    df = df.drop(columns=['Electtrical'])

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")

## Selecció de dades

Després de l'exploració anterior, pots decidir utilitzar o no utilitzar alguns dels conjunts de dades.

Per a aquest exercici **no hi ha cap decisió a prendre, utilitzes l'únic conjunt de dades que tenim**.

In [ ]:
# =====================================================================
# ## Selecció de dades
# =====================================================================
df_labeled = df[df['Split'] == 'labeled'].copy()
df_leaderboard = df[df['Split'] == 'leaderboard'].copy()

X_labeled = df_labeled.drop(columns=['Split', 'Id', 'SalePrice'])
y_labeled = df_labeled['SalePrice']
X_leaderboard = df_leaderboard.drop(columns=['Split', 'Id', 'SalePrice'])

# Split de validación para control
X_train, X_val, y_train, y_val = train_test_split(X_labeled, y_labeled, test_size=0.2, random_state=42)



## Neteja de dades

### Elimina les característiques innecessàries (si n'hi ha)

In [ ]:
# =====================================================================
# ## Neteja de dades (Parcial)
# =====================================================================
def neteja_parcial_pipeline(X):
    X_clean = X.copy()

    # Descartamos únicamente lo que carece de valor predictivo objetivo (nulos masivos >90%)
    if 'MiscFeature' in X_clean.columns:
        X_clean = X_clean.drop(columns=['MiscFeature'])

    # CONSERVACIÓN: Dejamos los nulos de Garajes y sistemas eléctricos sin tocar
    return X_clean

X_train_parcial = neteja_parcial_pipeline(X_train)
X_val_parcial = neteja_parcial_pipeline(X_val)
X_leaderboard_parcial = neteja_parcial_pipeline(X_leaderboard)


### Tracta els valors nuls o erronis (si n'hi ha)

### Tracta les files duplicades que siguin errors (si n'hi ha)

### Decideix què fer amb els outliers (si n'hi ha)

Normalment, durant aquest pas pots decidir eliminar o canviar alguns outliers. No obstant això, **per a aquest exercici no eliminis cap outlier**, deixa'ls tal com estan.

## Construcció de dades

Decideix si vols crear noves característiques a partir de les existents. Pots ser tan creatiu com vulguis.

## Integració de dades

Decideix si vols integrar dades d'altres fonts.

**Això no és necessari per als exercicis.**

In [ ]:
# =====================================================================
# ## Integració de dades
# =====================================================================
# No requerida para los ejercicios
X_train_int = X_train_parcial.copy()
X_val_int = X_val_parcial.copy()
X_leaderboard_int = X_leaderboard_parcial.copy()



## Enginyeria de característiques

### Codificació

Aplica les codificacions que consideris més apropiades per a les variables categòriques.

### Binning

Aplica binning a algunes columnes si ho consideres apropiat.

In [ ]:
# =====================================================================
# ## Enginyeria de característiques (Binning y Construcción)
# =====================================================================
# Nota: Nos saltamos la subsección ### Codificació para conservar el texto original

### Binning
def aplicar_binning(X):
    X_bin = X.copy()
    bins = [0, 1950, 1980, 2000, 2010, 2030]
    labels = [0, 1, 2, 3, 4]
    X_bin['YearBuilt_Binned'] = pd.cut(X_bin['YearBuilt'], bins=bins, labels=labels).astype(float)
    return X_bin

X_train_binned = aplicar_binning(X_train_int)
X_val_binned = aplicar_binning(X_val_int)
X_leaderboard_binned = aplicar_binning(X_leaderboard_int)

### Correlacions altes

Decideix què fer amb les característiques que estan molt altament correlacionades, si n'hi ha.

**Construción de datos**
Construcció de característiques

In [ ]:
### Construcció de característiques
def construir_caracteristiques(X):
    X_built = X.copy()
    X_built['AnosDesdeRemod'] = 2026 - X_built['YearRemodAdd']
    X_built['Total_Habitable_SF'] = X_built['1stFlrSF'] + X_built['2ndFlrSF'] + X_built['TotalBsmtSF']

    # Tratamiento seguro solo para evitar errores matemáticos directos en la suma de baños
    bsmt_baths = X_built['BsmtFullBath'].fillna(0)
    X_built['Total_Bathrooms'] = X_built['FullBath'] + bsmt_baths
    return X_built

X_train_feat = construir_caracteristiques(X_train_binned)
X_val_feat = construir_caracteristiques(X_val_binned)
X_leaderboard_feat = construir_caracteristiques(X_leaderboard_binned)



Formatar dades

In [ ]:
# =====================================================================
# ## Formatar dades (Reconstrucción y Exportación de CSV)
# =====================================================================
X_train_feat['Id'] = df_labeled.loc[X_train_feat.index, 'Id']
X_train_feat['SalePrice'] = y_labeled.loc[X_train_feat.index]
X_train_feat['Split'] = 'train'

X_val_feat['Id'] = df_labeled.loc[X_val_feat.index, 'Id']
X_val_feat['SalePrice'] = y_labeled.loc[X_val_feat.index]
X_val_feat['Split'] = 'validation'

X_leaderboard_feat['Id'] = df_leaderboard.loc[X_leaderboard_feat.index, 'Id']
X_leaderboard_feat['SalePrice'] = np.nan
X_leaderboard_feat['Split'] = 'leaderboard'

# Asegurar orden uniforme
columnas_ordenadas = X_train_feat.columns.tolist()
X_val_feat = X_val_feat[columnas_ordenadas]
X_leaderboard_feat = X_leaderboard_feat[columnas_ordenadas]

# DataFrame unificado listo
df_parcial_nan = pd.concat([X_train_feat, X_val_feat, X_leaderboard_feat], axis=0).reset_index(drop=True)

# Guardar la base de datos resultante
df_parcial_nan.to_csv('house_pricing_parcial_nan.csv', index=False)

print("--- Archivo 'house_pricing_parcial_nan.csv' exportado correctamente ---")
print(f"Dimensiones del DataFrame intermedio: {df_parcial_nan.shape}")